# 01 - Veri Keşfi ve Ön İnceleme

Bu notebook, toplanan video verilerinin ve etiketlenen veri setlerinin incelenmesini kapsar.

## İçerik
1. Video özellikleri (çözünürlük, FPS, süre)
2. Veri seti istatistikleri (sınıf dağılımı, bbox boyutları)
3. Örnek görüntüler ve etiketler
4. Veri kalitesi kontrolü

In [ ]:
# Google Colab için kurulum
# !pip install ultralytics opencv-python matplotlib seaborn
# !git clone https://github.com/YOUR_REPO/hatched-area-violation-detection.git
# %cd hatched-area-violation-detection

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

## 1. Video Özellikleri

In [ ]:
VIDEO_DIR = Path('data/videos/raw')

video_info = []
for vpath in sorted(VIDEO_DIR.glob('*.mp4')):
    cap = cv2.VideoCapture(str(vpath))
    info = {
        'name': vpath.name,
        'width': int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        'height': int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        'fps': cap.get(cv2.CAP_PROP_FPS),
        'frames': int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
    }
    info['duration_sec'] = info['frames'] / info['fps'] if info['fps'] > 0 else 0
    video_info.append(info)
    cap.release()

import pandas as pd
df_videos = pd.DataFrame(video_info)
print(f'Toplam video: {len(df_videos)}')
print(f'Toplam süre: {df_videos["duration_sec"].sum() / 60:.1f} dakika')
df_videos

## 2. Örnek Kareler

In [ ]:
# Her videodan 1 örnek kare
fig, axes = plt.subplots(1, min(len(video_info), 4), figsize=(20, 5))
if not isinstance(axes, np.ndarray):
    axes = [axes]

for ax, vinfo in zip(axes, video_info[:4]):
    cap = cv2.VideoCapture(str(VIDEO_DIR / vinfo['name']))
    mid_frame = vinfo['frames'] // 2
    cap.set(cv2.CAP_PROP_POS_FRAMES, mid_frame)
    ret, frame = cap.read()
    cap.release()
    if ret:
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(vinfo['name'])
    ax.axis('off')

plt.tight_layout()
plt.savefig('results/sample_frames.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Veri Seti İstatistikleri

Etiketlenen araç tespiti veri seti (YOLO format).

In [ ]:
DATASET_DIR = Path('data/datasets/vehicle_detection')
LABEL_DIR = DATASET_DIR / 'labels' / 'train'

CLASS_NAMES = {0: 'car', 1: 'bus', 2: 'truck', 3: 'motorcycle', 4: 'minibus'}

all_labels = []
for lbl_file in sorted(LABEL_DIR.glob('*.txt')):
    with open(lbl_file) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                cls_id, cx, cy, w, h = int(parts[0]), *map(float, parts[1:5])
                all_labels.append({
                    'class_id': cls_id,
                    'class_name': CLASS_NAMES.get(cls_id, f'cls_{cls_id}'),
                    'cx': cx, 'cy': cy, 'w': w, 'h': h,
                    'area': w * h,
                })

df_labels = pd.DataFrame(all_labels)
print(f'Toplam etiket: {len(df_labels)}')
print(f'\nSınıf dağılımı:')
print(df_labels['class_name'].value_counts())

In [ ]:
# Sınıf dağılımı grafiği
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
df_labels['class_name'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Sınıf Dağılımı')
axes[0].set_xlabel('Sınıf')
axes[0].set_ylabel('Sayı')

# Bbox boyut dağılımı
axes[1].scatter(df_labels['w'], df_labels['h'], alpha=0.3, s=10)
axes[1].set_title('Bbox Boyut Dağılımı (Normalized)')
axes[1].set_xlabel('Genişlik')
axes[1].set_ylabel('Yükseklik')

plt.tight_layout()
plt.savefig('results/dataset_statistics.png', dpi=150, bbox_inches='tight')
plt.show()